# Silver Layer: Fuel Prices Transformation

This notebook transforms raw fuel price data from Bronze to Silver layer.

## Transformations Applied:
- **Column Standardization**: All columns converted to snake_case
- **Price Validation**: Ensure prices are positive and within reasonable ranges (0-500 cents/liter)
- **Fuel Type Standardization**: Clean and categorize fuel types (E10, U91, U95, U98, Diesel, LPG, etc.)
- **Date Parsing**: Convert lastupdated to proper timestamp format
- **Data Quality Flags**: Identify missing critical fields (station, price, fuel type)
- **Deduplication**: Remove duplicate records based on station + fuel type + date

## Data Quality Validations:
- Count of records with invalid prices
- Count of records with missing station codes
- Fuel type distribution
- Price range statistics by fuel type
- Duplicate detection

In [0]:
# Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Table paths
BRONZE_TABLE = "mobility_ai.bronze.fuel_raw"
SILVER_TABLE = "mobility_ai.silver.fuel_prices"

print(f"Source: {BRONZE_TABLE}")
print(f"Target: {SILVER_TABLE}")

In [0]:
# Load bronze data
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze record count: {bronze_df.count()}")

# Standardize column names to snake_case
def to_snake_case(col_name):
    import re
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', col_name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    return re.sub(r'[\s-]+', '_', s2).lower()

# Rename all columns
for col in bronze_df.columns:
    bronze_df = bronze_df.withColumnRenamed(col, to_snake_case(col))

# Parse and validate price (convert to double, ensure positive)
silver_df = bronze_df.withColumn(
    "price_cents",
    F.expr("try_cast(price as double)")
)

# Standardize fuel type (trim whitespace, uppercase)
silver_df = silver_df.withColumn(
    "fuel_type_clean",
    F.upper(F.trim(F.col("fueltype")))
)

# Categorize fuel types into standard groups
silver_df = silver_df.withColumn(
    "fuel_category",
    F.when(F.col("fuel_type_clean").isin("E10", "UNLEADED", "U91"), "Regular Unleaded")
     .when(F.col("fuel_type_clean").isin("U95", "PREMIUM 95"), "Premium 95")
     .when(F.col("fuel_type_clean").isin("U98", "PREMIUM 98"), "Premium 98")
     .when(F.col("fuel_type_clean").isin("DIESEL", "DL"), "Diesel")
     .when(F.col("fuel_type_clean").isin("LPG", "AUTOGAS"), "LPG")
     .otherwise("Other")
)

# Parse lastupdated timestamp with correct format (DD/MM/YYYY HH:mm:ss)
silver_df = silver_df.withColumn(
    "last_updated_ts",
    F.to_timestamp(F.col("lastupdated"), "dd/MM/yyyy HH:mm:ss")
)

# Add data quality flags
silver_df = silver_df.withColumn(
    "has_valid_price",
    F.when(
        (F.col("price_cents").isNotNull()) & 
        (F.col("price_cents") > 0) & 
        (F.col("price_cents") <= 500),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_station",
    F.when(F.col("stationcode").isNotNull() & (F.length(F.col("stationcode")) > 0), True)
     .otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_timestamp",
    F.when(F.col("last_updated_ts").isNotNull(), True)
     .otherwise(False)
)

# Deduplicate based on stationcode + fueltype + date
window_spec = Window.partitionBy(
    "stationcode", 
    "fuel_type_clean", 
    F.to_date("last_updated_ts")
).orderBy(F.col("last_updated_ts").desc())

silver_df = silver_df.withColumn("row_num", F.row_number().over(window_spec))
silver_df = silver_df.filter(F.col("row_num") == 1).drop("row_num")

# Add processing metadata
silver_df = silver_df.withColumn("silver_processed_at", F.current_timestamp())

print(f"Silver record count after transformations: {silver_df.count()}")
display(silver_df.limit(10))

In [0]:
# Write to silver table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"✅ Successfully wrote {silver_df.count()} records to {SILVER_TABLE}")

In [0]:
# Validate silver data
validation_df = spark.table(SILVER_TABLE)

# Quality metrics
total_count = validation_df.count()
invalid_price = validation_df.filter(F.col("has_valid_price") == False).count()
missing_station = validation_df.filter(F.col("has_valid_station") == False).count()
invalid_timestamp = validation_df.filter(F.col("has_valid_timestamp") == False).count()

print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
print(f"Total Records: {total_count}")
print(f"Invalid Prices: {invalid_price} ({invalid_price/total_count*100:.2f}%)")
print(f"Missing Station Codes: {missing_station} ({missing_station/total_count*100:.2f}%)")
print(f"Invalid Timestamps: {invalid_timestamp} ({invalid_timestamp/total_count*100:.2f}%)")
print("=" * 60)

# Fuel category distribution
print("\nFuel Category Distribution:")
validation_df.groupBy("fuel_category").count().orderBy("fuel_category").show()

# Price statistics by fuel type
print("\nPrice Statistics by Fuel Category (cents/liter):")
validation_df.filter(F.col("has_valid_price") == True).groupBy("fuel_category").agg(
    F.min("price_cents").alias("min_price"),
    F.avg("price_cents").alias("avg_price"),
    F.max("price_cents").alias("max_price"),
    F.count("*").alias("record_count")
).orderBy("fuel_category").show()

# Show sample of flagged records
print("\nSample of records with data quality issues:")
validation_df.filter(
    (F.col("has_valid_price") == False) | 
    (F.col("has_valid_station") == False) | 
    (F.col("has_valid_timestamp") == False)
).select(
    "stationcode", "state", "fuel_type_clean", "price_cents", 
    "has_valid_price", "has_valid_station", "has_valid_timestamp"
).show(10, truncate=False)